In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [4]:
!pip install -q sentence-transformers datasets faiss-gpu

In [6]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader

import faiss

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")

print(dataset)

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv: reconstructing file:   0%|          |  0.00B /  147MB            

ML-Arxiv-Papers.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
        num_rows: 117592
    })
})


In [8]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']
{'Unnamed: 0.1': 0, 'Unnamed: 0': 0.0, 'title': 'Learning from compressed observations', 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\np

In [9]:
df = dataset["train"].to_pandas()

df = df[["title", "abstract"]].copy()

df = df.dropna(subset=["title", "abstract"])

df["title"] = df["title"].str.strip()
df["abstract"] = df["abstract"].str.strip()

df = df[
    (df["title"] != "") &
    (df["abstract"] != "")
]

df = df.drop_duplicates(subset=["title"])

df = df.reset_index(drop=True)

print("Total papers:", len(df))
print(df.head())

Total papers: 115458
                                               title  \
0              Learning from compressed observations   
1  Sensor Networks with Random Links: Topology De...   
2  The on-line shortest path problem under partia...   
3    A neural network approach to ordinal regression   
4   Parametric Learning and Monte Carlo Optimization   

                                            abstract  
0  The problem of statistical learning is to cons...  
1  In a sensor network, in practice, the communic...  
2  The on-line shortest path problem is considere...  
3  Ordinal regression is an important type of lea...  
4  This paper uncovers and explores the close rel...  


In [10]:
df = df.sample(n=2000, random_state=13)
df = df.reset_index(drop=True)

print("Selected papers:", len(df))

Selected papers: 2000


In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.25,
    random_state=13
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.60,
    random_state=13
)

print(len(train_df), len(val_df), len(test_df))

1500 200 300


In [12]:
train_examples = [
    InputExample(
        texts=[row["title"], row["abstract"]]
    )
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(
        texts=[row["title"], row["abstract"]]
    )
    for _, row in val_df.iterrows()
]

test_examples = [
    InputExample(
        texts=[row["title"], row["abstract"]]
    )
    for _, row in test_df.iterrows()
]

In [13]:
print("Train:", len(train_examples))
print("Validation:", len(val_examples))
print("Test:", len(test_examples))

print("\nExample:")
print("Query:", train_examples[0].texts[0])
print("Positive:", train_examples[0].texts[1][:300])

Train: 1500
Validation: 200
Test: 300

Example:
Query: ACE: An Actor Ensemble Algorithm for Continuous Control with Tree Search
Positive: In this paper, we propose an actor ensemble algorithm, named ACE, for
continuous control with a deterministic policy in reinforcement learning. In
ACE, we use actor ensemble (i.e., multiple actors) to search the global maxima
of the critic. Besides the ensemble perspective, we also formulate ACE in 


In [14]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Embedding dimension: 384


/tmp/ipykernel_3307/2995554107.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [16]:
test_queries = test_df["title"].tolist()
test_documents = test_df["abstract"].tolist()

query_embeddings = model.encode(
    test_queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

document_embeddings = model.encode(
    test_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [17]:
similarities = query_embeddings @ document_embeddings.T

top_1 = np.argmax(similarities, axis=1)

correct = np.arange(len(test_queries))

recall_at_1 = np.mean(top_1 == correct)

print("Baseline Recall@1:", recall_at_1)

Baseline Recall@1: 0.9466666666666667


In [18]:
!pip install -q openai

In [20]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

print("OpenAI API connection initialized successfully.")

OpenAI API connection initialized successfully.


In [21]:
sample_paper = train_df.iloc[0]

prompt = f"""
You are helping build a semantic search engine for scientific papers.

Based on the following research paper, generate one natural-language
search query that a researcher might use to find this paper.

The query must:
- Describe the research problem or objective.
- Use natural language.
- Avoid copying the paper title.
- Avoid mentioning the paper title or authors.
- Be concise (one sentence).

Title: {sample_paper['title']}

Abstract: {sample_paper['abstract']}

Return only the search query.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.7
)

sample_query = response.choices[0].message.content.strip()

print("Original paper title:")
print(sample_paper["title"])

print("\nGenerated search query:")
print(sample_query)

Original paper title:
ACE: An Actor Ensemble Algorithm for Continuous Control with Tree Search

Generated search query:
What are the advancements in reinforcement learning for continuous control using an ensemble of actors and tree search techniques?


In [26]:
def generate_search_query(title, abstract):

    prompt = f"""
    You are helping build a semantic search engine for scientific papers.

    Based on the following research paper, generate one natural-language
    search query that a researcher might use to find this paper.

    Requirements:
    - Describe the research problem or objective.
    - Use natural language.
    - Do not copy the paper title.
    - Do not mention the title or authors.
    - Be concise (one sentence).
    - Avoid simply listing technical keywords.
    - Do not begin every query with "How can", "What", or "How does".
    - Express the research problem from a user's perspective rather than summarizing the paper.

    Title: {title}

    Abstract: {abstract}

    Return only the search query.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content.strip()

In [27]:
test_query = generate_search_query(
    sample_paper["title"],
    sample_paper["abstract"]
)

print("Generated query:")
print(test_query)

Generated query:
Looking for an advanced reinforcement learning algorithm that improves continuous control in robotic simulations using an ensemble of actors and tree search techniques.


In [29]:
from tqdm.auto import tqdm
import time

train_df = train_df.copy()
train_df["query"] = None

for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):

    try:
        train_df.at[idx, "query"] = generate_search_query(
            row["title"],
            row["abstract"]
        )

    except Exception as e:
        print(f"Error at index {idx}: {e}")
        time.sleep(2)

successful_queries = train_df["query"].notna().sum()

print(f"Successfully generated queries: {successful_queries}")
print(f"Failed queries: {len(train_df) - successful_queries}")

  0%|          | 0/1500 [00:00<?, ?it/s]

Successfully generated queries: 1500
Failed queries: 0


In [30]:
train_df.to_csv(
    "train_data.csv",
    index=False
)

print("Training dataset saved successfully.")
print(f"Total training samples: {len(train_df)}")
print(f"Missing queries: {train_df['query'].isna().sum()}")

Training dataset saved successfully.
Total training samples: 1500
Missing queries: 0


In [31]:
for _, row in train_df.sample(5, random_state=13).iterrows():
    print(f"\nTitle: {row['title']}")
    print(f"Query: {row['query']}")


Title: Unsupervised Skill Discovery with Bottleneck Option Learning
Query: What are the methods for discovering diverse skills in environments without external rewards or supervision?

Title: End-to-end music source separation: is it possible in the waveform
  domain?
Query: Can waveform-based models effectively separate music sources while preserving all audio information, including phase?

Title: Stochastic Mirror Descent: Convergence Analysis and Adaptive Variants
  via the Mirror Stochastic Polyak Stepsize
Query: What are the convergence properties and adaptive stepsize strategies for stochastic mirror descent in convex optimization?

Title: Federated Optimization: Distributed Machine Learning for On-Device
  Intelligence
Query: I'm looking for research on optimizing machine learning models using distributed data from numerous mobile devices while minimizing communication rounds.

Title: Large Batch Experience Replay
Query: What are the theoretical foundations and practical improv

In [32]:
val_df = val_df.copy()
val_queries = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):

    query = generate_search_query(
        row["title"],
        row["abstract"]
    )

    val_queries.append(query)

val_df["query"] = val_queries

print(f"Validation queries generated: {len(val_queries)}")

  0%|          | 0/200 [00:00<?, ?it/s]

Validation queries generated: 200


In [33]:
test_df = test_df.copy()
test_queries = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    query = generate_search_query(
        row["title"],
        row["abstract"]
    )

    test_queries.append(query)

test_df["query"] = test_queries

print(f"Test queries generated: {len(test_queries)}")

  0%|          | 0/300 [00:00<?, ?it/s]

Test queries generated: 300


In [34]:
train_df.to_csv("train_data.csv", index=False)
val_df.to_csv("validation_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

print("All datasets saved successfully.")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

All datasets saved successfully.
Training samples: 1500
Validation samples: 200
Test samples: 300


In [35]:
from sentence_transformers import InputExample

train_examples = [
    InputExample(
        texts=[row["query"], row["abstract"]]
    )
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(
        texts=[row["query"], row["abstract"]]
    )
    for _, row in val_df.iterrows()
]

print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")

Training examples: 1500
Validation examples: 200


In [36]:
from sentence_transformers import SentenceTransformer

baseline_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Baseline and fine-tuning models loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Baseline and fine-tuning models loaded successfully.


In [37]:
from torch.utils.data import DataLoader
from sentence_transformers import losses

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

train_loss = losses.MultipleNegativesRankingLoss(
    model=model
)

print(f"Training batches: {len(train_dataloader)}")
print(f"Batch size: {train_dataloader.batch_size}")
print("Loss function: MultipleNegativesRankingLoss")

Training batches: 94
Batch size: 16
Loss function: MultipleNegativesRankingLoss


In [38]:
import numpy as np

test_queries = test_df["query"].tolist()
test_documents = test_df["abstract"].tolist()

query_embeddings = baseline_model.encode(
    test_queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

document_embeddings = baseline_model.encode(
    test_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

similarities = query_embeddings @ document_embeddings.T

print("Baseline embeddings generated successfully.")
print(f"Query embeddings shape: {query_embeddings.shape}")
print(f"Document embeddings shape: {document_embeddings.shape}")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Baseline embeddings generated successfully.
Query embeddings shape: (300, 384)
Document embeddings shape: (300, 384)


In [39]:
ranked_indices = np.argsort(-similarities, axis=1)

correct_indices = np.arange(len(test_queries))

ranks = np.argmax(
    ranked_indices == correct_indices[:, None],
    axis=1
) + 1

recall_at_1 = np.mean(ranks <= 1)
recall_at_5 = np.mean(ranks <= 5)
mrr = np.mean(1 / ranks)

print(f"Baseline Recall@1: {recall_at_1:.4f}")
print(f"Baseline Recall@5: {recall_at_5:.4f}")
print(f"Baseline MRR: {mrr:.4f}")

Baseline Recall@1: 0.9700
Baseline Recall@5: 0.9967
Baseline MRR: 0.9828


In [40]:
warmup_steps = int(len(train_dataloader) * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=warmup_steps,
    output_path="models/fine_tuned_minilm",
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [41]:
fine_tuned_query_embeddings = model.encode(
    test_queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

fine_tuned_document_embeddings = model.encode(
    test_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

fine_tuned_similarities = (
    fine_tuned_query_embeddings @ fine_tuned_document_embeddings.T
)

print("Fine-tuned embeddings generated successfully.")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Fine-tuned embeddings generated successfully.


In [42]:
ranked_indices = np.argsort(-fine_tuned_similarities, axis=1)

ranks = np.argmax(
    ranked_indices == correct_indices[:, None],
    axis=1
) + 1

fine_tuned_recall_at_1 = np.mean(ranks <= 1)
fine_tuned_recall_at_5 = np.mean(ranks <= 5)
fine_tuned_mrr = np.mean(1 / ranks)

print(f"Fine-tuned Recall@1: {fine_tuned_recall_at_1:.4f}")
print(f"Fine-tuned Recall@5: {fine_tuned_recall_at_5:.4f}")
print(f"Fine-tuned MRR: {fine_tuned_mrr:.4f}")

Fine-tuned Recall@1: 0.9800
Fine-tuned Recall@5: 0.9967
Fine-tuned MRR: 0.9878


In [43]:
import pandas as pd

results = pd.DataFrame({
    "Metric": ["Recall@1", "Recall@5", "MRR"],
    "Baseline": [
        recall_at_1,
        recall_at_5,
        mrr
    ],
    "Fine-tuned": [
        fine_tuned_recall_at_1,
        fine_tuned_recall_at_5,
        fine_tuned_mrr
    ]
})

results["Improvement"] = results["Fine-tuned"] - results["Baseline"]

results

,Metric,Baseline,Fine-tuned,Improvement
0,Recall@1,0.970000,0.980000,0.010000
1,Recall@5,0.996667,0.996667,0.000000
2,MRR,0.982791,0.987805,0.005014


In [44]:
all_documents = df["abstract"].tolist()

baseline_all_embeddings = baseline_model.encode(
    all_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

fine_tuned_all_embeddings = model.encode(
    all_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Full document collection encoded successfully.")
print(f"Document count: {len(all_documents)}")

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Full document collection encoded successfully.
Document count: 2000


In [45]:
correct_indices_full = df.index.get_indexer(test_df.index)

baseline_full_similarities = (
    query_embeddings @ baseline_all_embeddings.T
)

fine_tuned_full_similarities = (
    fine_tuned_query_embeddings @ fine_tuned_all_embeddings.T
)

print(f"Test queries: {len(test_queries)}")
print(f"Candidate documents: {len(all_documents)}")
print(f"Correct document mapping: {len(correct_indices_full)}")
print("Full-corpus similarity matrices generated.")

Test queries: 300
Candidate documents: 2000
Correct document mapping: 300
Full-corpus similarity matrices generated.


In [46]:
def calculate_metrics(similarities, correct_indices):
    ranked_indices = np.argsort(-similarities, axis=1)

    ranks = np.argmax(
        ranked_indices == correct_indices[:, None],
        axis=1
    ) + 1

    return {
        "Recall@1": np.mean(ranks <= 1),
        "Recall@5": np.mean(ranks <= 5),
        "MRR": np.mean(1 / ranks)
    }


baseline_full_metrics = calculate_metrics(
    baseline_full_similarities,
    correct_indices_full
)

fine_tuned_full_metrics = calculate_metrics(
    fine_tuned_full_similarities,
    correct_indices_full
)

full_results = pd.DataFrame({
    "Metric": baseline_full_metrics.keys(),
    "Baseline": baseline_full_metrics.values(),
    "Fine-tuned": fine_tuned_full_metrics.values()
})

full_results["Improvement"] = (
    full_results["Fine-tuned"] - full_results["Baseline"]
)

full_results

,Metric,Baseline,Fine-tuned,Improvement
0,Recall@1,0.910000,0.933333,0.023333
1,Recall@5,0.983333,0.990000,0.006667
2,MRR,0.941923,0.959670,0.017748


In [47]:
import faiss

embedding_dimension = fine_tuned_all_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(
    fine_tuned_all_embeddings.astype("float32")
)

print(f"FAISS index created successfully.")
print(f"Indexed documents: {faiss_index.ntotal}")
print(f"Embedding dimension: {embedding_dimension}")

FAISS index created successfully.
Indexed documents: 2000
Embedding dimension: 384


In [48]:
def semantic_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        query_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": df.iloc[idx]["title"],
            "abstract": df.iloc[idx]["abstract"],
            "similarity": float(score)
        })

    return results

In [49]:
results = semantic_search(
    "How can deep learning models improve medical image classification?",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']}")
    print(f"Similarity: {result['similarity']:.4f}")


1. In-Line Image Transformations for Imbalanced, Multiclass Computer Vision
  Classification of Lung Chest X-Rays
Similarity: 0.5753

2. A Survey on Deep Learning and Explainability for Automatic Report
  Generation from Medical Images
Similarity: 0.5665

3. A Light CNN for detecting COVID-19 from CT scans of the chest
Similarity: 0.5536

4. Semi-Supervised Learning for Image Classification using Compact Networks
  in the BioMedical Context
Similarity: 0.5423

5. Estimating Uncertainty and Interpretability in Deep Learning for
  Coronavirus (COVID-19) Detection
Similarity: 0.5380


In [50]:
def build_context(results):
    context = ""

    for i, result in enumerate(results, 1):
        context += f"""
Paper {i}
Title: {result['title']}
Abstract: {result['abstract']}
Similarity: {result['similarity']:.4f}

"""

    return context

In [58]:
def generate_answer(query, context):

    prompt = f"""
You are an academic research assistant.

Answer the user's question using only the scientific papers provided below.

Instructions:
- Base your answer strictly on the provided abstracts.
- Do not invent findings or information.
- Cite relevant papers using their titles.
- If the papers do not contain enough information, state this clearly.
- Write a clear and concise answer.
- Do not suggest that a method can be adapted to another domain unless the provided papers explicitly support this claim.
- Clearly distinguish reported findings from your own interpretation.
- Cite papers using their exact titles, not generic labels such as "Paper 1".
- Only cite a paper when its content directly supports the relevant statement.

Retrieved Papers:
{context}

User Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a reliable academic research assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content.strip()

In [59]:
def rag_pipeline(query, top_k=5):

    retrieved_results = semantic_search(
        query,
        top_k=top_k
    )

    context = build_context(retrieved_results)

    answer = generate_answer(
        query,
        context
    )

    return {
        "query": query,
        "answer": answer,
        "sources": retrieved_results
    }

In [60]:
query = "What deep learning approaches are used for detecting diseases from medical images?"

response = rag_pipeline(query, top_k=5)

print("Question:")
print(response["query"])

print("\nGenerated Answer:")
print(response["answer"])

print("\nRetrieved Sources:")

for i, source in enumerate(response["sources"], 1):
    print(f"\n{i}. {source['title']}")
    print(f"Similarity: {source['similarity']:.4f}")

Question:
What deep learning approaches are used for detecting diseases from medical images?

Generated Answer:
Several deep learning approaches have been utilized for detecting diseases from medical images, as highlighted in the provided papers:

1. **Convolutional Neural Networks (CNNs)**: CNNs are prominently used for classifying medical images, such as lung chest X-rays (LCXRs) and CT scans. For instance, in "In-Line Image Transformations for Imbalanced, Multiclass Computer Vision Classification of Lung Chest X-Rays," a simple CNN architecture achieved 94% accuracy in multiclass LCXR classification. Similarly, "A Light CNN for detecting COVID-19 from CT scans of the chest" describes a modified SqueezeNet CNN that achieved 83% accuracy in distinguishing COVID-19 CT images from other types.

2. **Unsupervised Deep Learning Techniques**: The paper "Unsupervised deep learning techniques for powdery mildew recognition based on multispectral imaging" discusses the use of unsupervised dee

In [61]:
import os

os.makedirs("models", exist_ok=True)

model.save("models/fine_tuned_minilm")

faiss.write_index(
    faiss_index,
    "models/papers_faiss.index"
)

print("Model and FAISS index saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and FAISS index saved successfully.


In [62]:
os.makedirs("data", exist_ok=True)

train_df.to_csv("data/train_data.csv", index=False)
val_df.to_csv("data/validation_data.csv", index=False)
test_df.to_csv("data/test_data.csv", index=False)
df.to_csv("data/all_papers.csv", index=False)

print("All datasets saved successfully.")
print(f"Training papers: {len(train_df)}")
print(f"Validation papers: {len(val_df)}")
print(f"Test papers: {len(test_df)}")
print(f"Total papers: {len(df)}")

All datasets saved successfully.
Training papers: 1500
Validation papers: 200
Test papers: 300
Total papers: 2000


In [63]:
for root, dirs, files in os.walk("."):
    if root.startswith("./models") or root.startswith("./data"):
        for file in files:
            path = os.path.join(root, file)
            print(f"{path} - {os.path.getsize(path) / 1024:.2f} KB")

./data/test_data.csv - 408.82 KB
./data/validation_data.csv - 269.45 KB
./data/all_papers.csv - 2429.15 KB
./data/train_data.csv - 2012.88 KB
./models/papers_faiss.index - 3000.04 KB
./models/fine_tuned_minilm/tokenizer_config.json - 0.58 KB
./models/fine_tuned_minilm/tokenizer.json - 694.97 KB
./models/fine_tuned_minilm/sentence_bert_config.json - 0.24 KB
./models/fine_tuned_minilm/config.json - 0.73 KB
./models/fine_tuned_minilm/config_sentence_transformers.json - 0.28 KB
./models/fine_tuned_minilm/model.safetensors - 88734.56 KB
./models/fine_tuned_minilm/README.md - 36.74 KB
./models/fine_tuned_minilm/modules.json - 0.42 KB
./models/fine_tuned_minilm/1_Pooling/config.json - 0.09 KB


In [64]:
import shutil

shutil.make_archive(
    "academic_paper_assistant",
    "zip",
    ".",
    "data"
)

print("Project data archive created.")

Project data archive created.


In [65]:
import os
import shutil

project_dir = "academic_paper_assistant"

os.makedirs(project_dir, exist_ok=True)

shutil.copytree("data", f"{project_dir}/data", dirs_exist_ok=True)
shutil.copytree("models", f"{project_dir}/models", dirs_exist_ok=True)

print("Project folders copied successfully.")

Project folders copied successfully.


In [66]:
shutil.make_archive(
    "academic_paper_assistant",
    "zip",
    root_dir=".",
    base_dir=project_dir
)

print("Complete project archive created.")

Complete project archive created.
